In [1]:
"""
E-commerce funnel analysis (event-level data -> session-level metrics).

Usage:  python analysis.py [path/to/day1_ecommerce_funnel_events.csv] [output_dir]

Key rule: every funnel stage is counted with DISTINCT session_id, never with row counts.
Overall conversion = sessions with a purchase / sessions with a product view * 100.
"""
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

# TASK 1 - DATA QUALITY

In [2]:
# read csv file
df = pd.read_csv('/content/day1_ecommerce_funnel_events.csv')

In [3]:
import sys
from pathlib import Path

# Detect if running in an interactive IPython/Colab environment
try:
    is_notebook = 'ipykernel_launcher' in sys.argv[0] or any('ipykernel' in arg or arg.endswith('.json') for arg in sys.argv)
except Exception:
    is_notebook = False

if is_notebook:
    CSV = 'day1_ecommerce_funnel_events.csv'
    OUT = Path('output')
else:
    CSV = sys.argv[1] if len(sys.argv) > 1 else 'day1_ecommerce_funnel_events.csv'
    OUT = Path(sys.argv[2]) if len(sys.argv) > 2 else Path('output')

OUT.mkdir(parents=True, exist_ok=True)

STAGES = ["product_view", "add_to_cart", "payment_selected", "purchase"]
STAGE_NO = {s: i + 1 for i, s in enumerate(STAGES)}
BLUE, RED, GREY = "#2a6fdb", "#d1495b", "#9aa5b1"

In [4]:
df.head(1)

,event_id,session_id,customer_id,event_timestamp,country,device,product_category,payment_method,event_type,order_value_eur
0,E000001,S00001,C0063,2026-09-24 12:38:00,Germany,Desktop,Accessories,NaN,product_view,0.0


In [5]:
df = pd.read_csv(CSV)
print("=" * 70, "\nTASK 1 - DATA QUALITY\n" + "=" * 70)
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes.to_string())
print("\nMissing Values:\n", df.isnull().sum().to_string())
print("\nDuplicate event_id:", df.event_id.duplicated().sum(),
      "| fully duplicated rows:", df.duplicated().sum())
print("Unique sessions:", df.session_id.nunique(), "| unique customers:", df.customer_id.nunique())
print("Date range:", df.event_timestamp.min(), "->", df.event_timestamp.max())

for c in ["country", "device", "product_category", "payment_method", "event_type"]:
  print(f"\nValues of {c}:")
  print(df[c].value_counts().to_string())

df["stage_no"] = df.event_type.map(STAGE_NO)
print("\nUnknown event types:", df.stage_no.isna().sum())
print("Negative order values:", (df.order_value_eur < 0).sum())
print("Order value by event type (should be >0 only for purchase):")
print(df.groupby("event_type").order_value_eur.agg(["min", "max", lambda s: (s > 0).sum()])
      .rename(columns={"<lambda_0>": "rows_>0"}).to_string())
print("\nPayment method present by event type (expected: only payment_selected & purchase):")
print(df.groupby("event_type").payment_method.value_counts().unstack().fillna(0).astype(int).to_string())
print("-> the 1,486 'missing' payment methods are structural (not yet chosen), not errors.")


TASK 1 - DATA QUALITY
Shape: (2161, 10)

Data Types:
 event_id             object
session_id           object
customer_id          object
event_timestamp      object
country              object
device               object
product_category     object
payment_method       object
event_type           object
order_value_eur     float64

Missing Values:
 event_id               0
session_id             0
customer_id            0
event_timestamp        0
country                0
device                 0
product_category       0
payment_method      1486
event_type             0
order_value_eur        0

Duplicate event_id: 0 | fully duplicated rows: 0
Unique sessions: 900 | unique customers: 499
Date range: 2026-09-01 00:20:00 -> 2026-09-30 23:28:00

Values of country:
country
Germany        1079
France          304
Netherlands     284
Italy           248
Spain           246

Values of device:
device
Mobile     1389
Desktop     649
Tablet      123

Values of product_category:
product_category


In [6]:
# Session consistency + logical sequences
for c in ["customer_id", "country", "device", "product_category"]:
    print(f"Sessions with >1 {c}:", (df.groupby("session_id")[c].nunique() > 1).sum())
print("Customers seen in >1 country:", (df.groupby("customer_id").country.nunique() > 1).sum(),
      "(sessions are consistent, but customer_id may not be a reliable person key)")

ordered = df.sort_values(["session_id", "event_timestamp", "stage_no"])
seq = ordered.groupby("session_id").event_type.apply(" > ".join)
print("\nSession event sequences:\n", seq.value_counts().to_string())
stage_sets = df.groupby("session_id").stage_no.apply(set)
print("\nSessions with a skipped stage:",
      stage_sets.apply(lambda s: any(k not in s for k in range(1, max(s)))).sum())
print("Sessions without a product_view:", stage_sets.apply(lambda s: 1 not in s).sum())
print("Repeated stage events within a session:",
      (df.groupby(["session_id", "event_type"]).size() > 1).sum())
by_time = df.sort_values(["session_id", "event_timestamp"])
print("Sessions whose timestamp order contradicts stage order:",
      by_time.groupby("session_id").stage_no.apply(lambda x: (x.diff().dropna() < 0).any()).sum())

Sessions with >1 customer_id: 0
Sessions with >1 country: 0
Sessions with >1 device: 0
Sessions with >1 product_category: 0
Customers seen in >1 country: 197 (sessions are consistent, but customer_id may not be a reliable person key)

Session event sequences:
 event_type
product_view                                                314
product_view > add_to_cart > payment_selected > purchase    250
product_view > add_to_cart > payment_selected               175
product_view > add_to_cart                                  161

Sessions with a skipped stage: 0
Sessions without a product_view: 0
Repeated stage events within a session: 0
Sessions whose timestamp order contradicts stage order: 0


## Build SESSION-level table (one row per session)


In [7]:
flags = (df.pivot_table(index="session_id", columns="event_type",
                        values="event_id", aggfunc="count").notna())[STAGES]
sessions = flags.join(df.groupby("session_id")[["country", "device", "product_category"]].first())
sessions["payment_method"] = (df.dropna(subset=["payment_method"])
                              .groupby("session_id").payment_method.first())
sessions["revenue"] = df.groupby("session_id").order_value_eur.sum()


In [8]:
sessions.head(1)

,product_view,add_to_cart,payment_selected,purchase,country,device,product_category,payment_method,revenue
session_id,,,,,,,,,
S00001,True,False,False,False,Germany,Desktop,Accessories,NaN,0.0


In [9]:
df.head(1)

,event_id,session_id,customer_id,event_timestamp,country,device,product_category,payment_method,event_type,order_value_eur,stage_no
0,E000001,S00001,C0063,2026-09-24 12:38:00,Germany,Desktop,Accessories,NaN,product_view,0.0,1


# TASK 2 - FUNNEL KPIs


In [11]:
print("\n" + "=" * 70, "\nTASK 2 - FUNNEL KPIs (distinct sessions)\n" + "=" * 70)
funnel = sessions[STAGES].sum().astype(int)
rows = []
for i, s in enumerate(STAGES):
    prev = funnel.iloc[i - 1] if i else np.nan
    rows.append({"stage": s, "sessions": funnel[s],
                 "step_conv_%": funnel[s] / prev * 100 if i else np.nan,
                 "dropped_sessions": prev - funnel[s] if i else np.nan,
                 "pct_of_views_%": funnel[s] / funnel.iloc[0] * 100})
funnel_tbl = pd.DataFrame(rows)
print(funnel_tbl.round(2).to_string(index=False))

overall = funnel["purchase"] / funnel["product_view"] * 100
print(f"\nOverall purchase conversion = {funnel['purchase']} / {funnel['product_view']} x 100 = {overall:.2f}%")
print(f"Row count for comparison: {len(df)} rows vs {df.session_id.nunique()} sessions")
revenue = sessions.revenue.sum()
print(f"Revenue: EUR {revenue:,.2f} | AOV: EUR {revenue / funnel['purchase']:.2f} "
      f"| revenue per product-view session: EUR {revenue / funnel['product_view']:.2f}")
funnel_tbl.to_csv(OUT / "funnel_kpis.csv", index=False)


TASK 2 - FUNNEL KPIs (distinct sessions)
           stage  sessions  step_conv_%  dropped_sessions  pct_of_views_%
    product_view       900          NaN               NaN          100.00
     add_to_cart       586        65.11             314.0           65.11
payment_selected       425        72.53             161.0           47.22
        purchase       250        58.82             175.0           27.78

Overall purchase conversion = 250 / 900 x 100 = 27.78%
Row count for comparison: 2161 rows vs 900 sessions
Revenue: EUR 11,097.46 | AOV: EUR 44.39 | revenue per product-view session: EUR 12.33


In [ ]:
print("\n" + "=" * 70, "\nTASK 3 - SEGMENT ANALYSIS\n" + "=" * 70)


def seg_table(col):
    t = sessions.groupby(col)[STAGES].sum()
    out = pd.DataFrame({
        "product_view": t.product_view,
        "purchases": t.purchase,
        "view_to_cart_%": t.add_to_cart / t.product_view * 100,
        "cart_to_payment_%": t.payment_selected / t.add_to_cart * 100,
        "payment_to_purchase_%": t.purchase / t.payment_selected * 100,
        "overall_conv_%": t.purchase / t.product_view * 100,
    })
    # Wilson 95% interval for overall conversion (shows how uncertain small segments are)
    n, k, z = out.product_view, out.purchases, 1.96
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    out["ci95_low_%"], out["ci95_high_%"] = (centre - half) * 100, (centre + half) * 100
    chi_p = chi2_contingency(np.c_[k, n - k])[1]  # converted vs not, across segments
    return out.sort_values("overall_conv_%", ascending=False), chi_p


seg_results = {}
for col in ["device", "country", "product_category"]:
    tbl, p = seg_table(col)
    seg_results[col] = tbl
    print(f"\n--- {col} (denominator = sessions with a product view) ---")
    print(tbl.round(1).to_string())
    print(f"Chi-square test, differences across {col}: p = {p:.3f}",
          "(statistically significant)" if p < 0.05 else "(NOT significant at 5% - treat ranking as indicative)")
    print(f"Strongest: {tbl.index[0]} ({tbl.iloc[0]['overall_conv_%']:.1f}%) | "
          f"Weakest: {tbl.index[-1]} ({tbl.iloc[-1]['overall_conv_%']:.1f}%)")
    tbl.to_csv(OUT / f"segment_{col}.csv")

# Payment method: recorded only from payment_selected onwards
print("\n--- payment_method (CAUTION) ---")
print("Payment method is only recorded at payment_selected and purchase. Sessions that\n"
      "never reached payment selection have no method, so it CANNOT be compared on the\n"
      "all-visitor denominator. The only fair metric is payment_selected -> purchase,\n"
      "and it describes a self-selected, already-engaged group (not all visitors).")
pm = (sessions.dropna(subset=["payment_method"]).groupby("payment_method")
      .agg(sessions_selected=("purchase", "size"), purchases=("purchase", "sum")))
pm["selected_to_purchase_%"] = pm.purchases / pm.sessions_selected * 100
pm["share_of_selections_%"] = pm.sessions_selected / pm.sessions_selected.sum() * 100
n, k, z = pm.sessions_selected, pm.purchases, 1.96
p_ = k / n
centre = (p_ + z**2 / (2 * n)) / (1 + z**2 / n)
half = z * np.sqrt(p_ * (1 - p_) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
pm["ci95_low_%"], pm["ci95_high_%"] = (centre - half) * 100, (centre + half) * 100
pm = pm.sort_values("selected_to_purchase_%", ascending=False)
print(pm.round(1).to_string())
chi_p = chi2_contingency(np.c_[pm.purchases, pm.sessions_selected - pm.purchases])[1]
print(f"Chi-square test across payment methods: p = {chi_p:.3f}")
pm.to_csv(OUT / "segment_payment_method.csv")

# Biggest drop-off: stage and segment
drops = funnel_tbl.dropna().assign(drop_rate=lambda d: 100 - d["step_conv_%"])
worst = drops.sort_values("dropped_sessions", ascending=False).iloc[0]
print(f"\nLargest absolute drop: {STAGES[STAGES.index(worst.stage) - 1]} -> {worst.stage} "
      f"({int(worst.dropped_sessions)} sessions)")
worst_rate = drops.sort_values("drop_rate", ascending=False).iloc[0]
print(f"Highest drop RATE: ... -> {worst_rate.stage} ({worst_rate.drop_rate:.1f}% lost)")


# TASK 3 - SEGMENT ANALYSIS


In [ ]:
sessions.head(1)

,product_view,add_to_cart,payment_selected,purchase,country,device,product_category,payment_method,revenue
session_id,,,,,,,,,
S00001,True,False,False,False,Germany,Desktop,Accessories,NaN,0.0


In [ ]:
print("\n" + "=" * 70, "\nTASK 3 - SEGMENT ANALYSIS\n" + "=" * 70)


def seg_table(col):
    t = sessions.groupby(col)[STAGES].sum()
    out = pd.DataFrame({
        "product_view": t.product_view,
        "purchases": t.purchase,
        "view_to_cart_%": t.add_to_cart / t.product_view * 100,
        "cart_to_payment_%": t.payment_selected / t.add_to_cart * 100,
        "payment_to_purchase_%": t.purchase / t.payment_selected * 100,
        "overall_conv_%": t.purchase / t.product_view * 100,
    })
    # Wilson 95% interval for overall conversion (shows how uncertain small segments are)
    n, k, z = out.product_view, out.purchases, 1.96
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    out["ci95_low_%"], out["ci95_high_%"] = (centre - half) * 100, (centre + half) * 100
    chi_p = chi2_contingency(np.c_[k, n - k])[1]  # converted vs not, across segments
    return out.sort_values("overall_conv_%", ascending=False), chi_p


seg_results = {}
for col in ["device", "country", "product_category"]:
    tbl, p = seg_table(col)
    seg_results[col] = tbl
    print(f"\n--- {col} (denominator = sessions with a product view) ---")
    print(tbl.round(1).to_string())
    print(f"Chi-square test, differences across {col}: p = {p:.3f}",
          "(statistically significant)" if p < 0.05 else "(NOT significant at 5% - treat ranking as indicative)")
    print(f"Strongest: {tbl.index[0]} ({tbl.iloc[0]['overall_conv_%']:.1f}%) | "
          f"Weakest: {tbl.index[-1]} ({tbl.iloc[-1]['overall_conv_%']:.1f}%)")
    tbl.to_csv(OUT / f"segment_{col}.csv")

# Payment method: recorded only from payment_selected onwards
print("\n--- payment_method (CAUTION) ---")
print("Payment method is only recorded at payment_selected and purchase. Sessions that\n"
      "never reached payment selection have no method, so it CANNOT be compared on the\n"
      "all-visitor denominator. The only fair metric is payment_selected -> purchase,\n"
      "and it describes a self-selected, already-engaged group (not all visitors).")
pm = (sessions.dropna(subset=["payment_method"]).groupby("payment_method")
      .agg(sessions_selected=("purchase", "size"), purchases=("purchase", "sum")))
pm["selected_to_purchase_%"] = pm.purchases / pm.sessions_selected * 100
pm["share_of_selections_%"] = pm.sessions_selected / pm.sessions_selected.sum() * 100
n, k, z = pm.sessions_selected, pm.purchases, 1.96
p_ = k / n
centre = (p_ + z**2 / (2 * n)) / (1 + z**2 / n)
half = z * np.sqrt(p_ * (1 - p_) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
pm["ci95_low_%"], pm["ci95_high_%"] = (centre - half) * 100, (centre + half) * 100
pm = pm.sort_values("selected_to_purchase_%", ascending=False)
print(pm.round(1).to_string())
chi_p = chi2_contingency(np.c_[pm.purchases, pm.sessions_selected - pm.purchases])[1]
print(f"Chi-square test across payment methods: p = {chi_p:.3f}")
pm.to_csv(OUT / "segment_payment_method.csv")

# Biggest drop-off: stage and segment
drops = funnel_tbl.dropna().assign(drop_rate=lambda d: 100 - d["step_conv_%"])
worst = drops.sort_values("dropped_sessions", ascending=False).iloc[0]
print(f"\nLargest absolute drop: {STAGES[STAGES.index(worst.stage) - 1]} -> {worst.stage} "
      f"({int(worst.dropped_sessions)} sessions)")
worst_rate = drops.sort_values("drop_rate", ascending=False).iloc[0]
print(f"Highest drop RATE: ... -> {worst_rate.stage} ({worst_rate.drop_rate:.1f}% lost)")



TASK 3 - SEGMENT ANALYSIS

--- device (denominator = sessions with a product view) ---
         product_view  purchases  view_to_cart_%  cart_to_payment_%  payment_to_purchase_%  overall_conv_%  ci95_low_%  ci95_high_%
device                                                                                                                             
Tablet             46         17            76.1               71.4                   68.0            37.0        24.5         51.4
Desktop           258         85            67.4               75.9                   64.4            32.9        27.5         38.9
Mobile            596        148            63.3               71.1                   55.2            24.8        21.5         28.5
Chi-square test, differences across device: p = 0.019 (statistically significant)
Strongest: Tablet (37.0%) | Weakest: Mobile (24.8%)

--- country (denominator = sessions with a product view) ---
             product_view  purchases  view_to_cart_%  ca

# TASK 4 - VISUALIZATIONS


In [ ]:
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

# 4.1 Funnel chart (centered horizontal bars)
fig, ax = plt.subplots(figsize=(9, 5))
labels = ["Product view", "Add to cart", "Payment selected", "Purchase"]
vals = funnel.values
for i, v in enumerate(vals):
    ax.barh(len(vals) - 1 - i, v, left=(vals[0] - v) / 2, color=BLUE, alpha=1 - 0.12 * i, height=0.85)
    pct = f"{v / vals[0] * 100:.0f}% of views"
    ax.text(vals[0] / 2, len(vals) - 1 - i, f"{labels[i]}\n{v:,} sessions\n({pct})",
            ha="center", va="center", color="white", fontweight="bold", fontsize=10)
for i in range(1, len(vals)):
    ax.text(vals[0] * 1.02, len(vals) - 1 - i + 0.5, f"{vals[i] / vals[i - 1] * 100:.1f}% continue",
            va="center", color="#333", fontsize=9)
ax.set_xlim(0, vals[0] * 1.25)
ax.axis("off")
ax.set_title(f"Session funnel - overall conversion {overall:.1f}% (distinct sessions)", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT / "1_funnel.png", dpi=160)
plt.close(fig)

# 4.2 Segment comparison (overall conversion, with 95% CI and overall average line)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharey=False)
for ax, col in zip(axes, ["device", "country", "product_category"]):
    t = seg_results[col].sort_values("overall_conv_%")
    err = [t["overall_conv_%"] - t["ci95_low_%"], t["ci95_high_%"] - t["overall_conv_%"]]
    colors = [RED if v == t["overall_conv_%"].min() else (BLUE if v == t["overall_conv_%"].max() else GREY)
              for v in t["overall_conv_%"]]
    ax.barh(t.index, t["overall_conv_%"], xerr=err, color=colors, ecolor="#555", capsize=3)
    ax.axvline(overall, color="black", ls="--", lw=1)
    for y, (v, n_) in enumerate(zip(t["overall_conv_%"], t.product_view)):
        ax.text(1, y, f"{v:.1f}%  (n={n_})", va="center", color="white", fontweight="bold", fontsize=9)
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("Purchase conversion %")
axes[0].text(overall, -0.85, f" avg {overall:.1f}%", fontsize=8)
fig.suptitle("Overall conversion by segment (sessions with a purchase / sessions with a product view; bars = 95% CI)",
             fontweight="bold")
fig.tight_layout()
fig.savefig(OUT / "2_segment_comparison.png", dpi=160)
plt.close(fig)

# 4.3 Biggest drop-off chart
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
step_names = ["View -> Cart", "Cart -> Payment", "Payment -> Purchase"]
lost = (funnel.values[:-1] - funnel.values[1:])
lost_rate = lost / funnel.values[:-1] * 100
big = int(np.argmax(lost))
cols = [RED if i == big else GREY for i in range(3)]
axes[0].bar(step_names, lost, color=cols)
for i, (l, r) in enumerate(zip(lost, lost_rate)):
    axes[0].text(i, l + 4, f"{l} sessions\n({r:.1f}% lost)", ha="center", fontsize=10)
axes[0].set_ylim(0, lost.max() * 1.25)
axes[0].set_title("Sessions lost at each step", fontweight="bold")
axes[0].set_ylabel("Sessions")

# Payment->purchase conversion by device x payment step (largest weak spot: late-stage)
dp = sessions.groupby("device")[STAGES].sum()
late = (dp.purchase / dp.payment_selected * 100).sort_values()
axes[1].barh(late.index, late.values, color=[RED] + [GREY] * (len(late) - 1))
axes[1].axvline(funnel.purchase / funnel.payment_selected * 100, color="black", ls="--", lw=1)
for y, v in enumerate(late.values):
    axes[1].text(1, y, f"{v:.1f}%", va="center", color="white", fontweight="bold")
axes[1].set_title("Payment -> purchase completion by device", fontweight="bold")
axes[1].set_xlabel("% of payment-selected sessions that purchase (dashed = average)")
fig.tight_layout()
fig.savefig(OUT / "3_biggest_dropoff.png", dpi=160)
plt.close(fig)

# 4.4 Payment method (separate: different denominator)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
t = pm.sort_values("selected_to_purchase_%")
err = [t["selected_to_purchase_%"] - t["ci95_low_%"], t["ci95_high_%"] - t["selected_to_purchase_%"]]
ax.barh(t.index, t["selected_to_purchase_%"], xerr=err, color=GREY, ecolor="#555", capsize=3)
for y, (v, n_) in enumerate(zip(t["selected_to_purchase_%"], t.sessions_selected)):
    ax.text(1, y, f"{v:.1f}%  (n={n_})", va="center", color="white", fontweight="bold")
ax.set_xlabel("Purchase rate among sessions that SELECTED a payment method (%)")
ax.set_title("Payment method: selected -> purchase\n(not comparable to all-visitor conversion)", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT / "4_payment_method.png", dpi=160)
plt.close(fig)

print(f"\nSaved charts and tables to: {OUT.resolve()}")



Saved charts and tables to: /content/output
